# SmartGuard-Pro — Analisi degli infortuni sul lavoro in Italia

**Autore:** Geson Bani  
**Data:** luglio 2026  
**Descrizione:** Analisi degli infortuni sul lavoro in Italia per settore, 
regione e anno, a supporto delle decisioni di investimento per un dispositivo 
di sicurezza (Smart PPE).

---

## 1. Obiettivo e domande di analisi

Questo progetto nasce a supporto di **Smart PPE**, un dispositivo di sicurezza 
per l'ambiente di lavoro. L'analisi dati in questo notebook serve a individuare 
i settori e le regioni con maggiore incidenza e costo degli infortuni, per 
orientare le decisioni di investimento.

> **Nota:** questo notebook si concentra sull'**analisi dei dati**. La 
> descrizione del prodotto Smart PPE e la presentazione dei risultati agli 
> investitori sono trattate nella dashboard finale (Power BI), che unisce 
> analisi di mercato e scheda prodotto.

L'obiettivo è capire **dove** e **in quali settori** si concentrano gli 
infortuni sul lavoro in Italia, e **quanto costano** alle aziende, così da 
individuare i comparti su cui un investimento in prevenzione avrebbe maggiore 
impatto.

Per farlo l'analisi incrocia tre tipi di dati:
- **infortuni** (INAIL) — quanti e dove
- **imprese** (ISTAT ASIA) — quante aziende e addetti per settore
- **retribuzioni** (INPS) — il costo del lavoro per settore

**Domande a cui l'analisi vuole rispondere:**
1. Quanti infortuni si registrano per settore, regione e anno?
2. Come si distribuisce il rischio una volta rapportato al numero di imprese/addetti?
3. Quali comparti combinano alta frequenza di infortuni e alti costi?
4. Su quali settori/regioni conviene indirizzare gli investimenti in prevenzione?

## 2. Setup e prerequisiti

In questa sezione importo le librerie utilizzate nel progetto. Il blocco 
verrà integrato man mano che il notebook cresce, aggiungendo le librerie nel 
momento in cui servono.

In [1]:
import requests                    # scaricare dati e file dal web
import os                          # gestione file e cartelle
import pandas as pd                # manipolazione e analisi delle tabelle
import matplotlib.pyplot as plt    # grafici
import seaborn as sns              # grafici statistici
import zipfile

## 3. Le fonti dati

L'analisi combina tre fonti pubbliche. La tabella riassume cosa fornisce 
ciascuna, in che formato e con quale frequenza di aggiornamento.

| Fonte | Cosa fornisce | Formato | Aggiornamento |
|-------|---------------|---------|---------------|
| INAIL | Infortuni sul lavoro (per regione, settore, anno) | CSV (zip) | Semestrale (luglio / dicembre) |
| ISTAT ASIA | ... | ... | ... |
| INPS | ... | ... | ... |

## 4. Acquisizione dati

In questa sezione vengono reperiti i dati grezzi dalle diverse fonti. Per 
ciascuna fonte si documenta: la valutazione che ha portato a scegliere quella 
fonte e quel formato, la struttura di accesso ai file, il download e la verifica 
dei dati scaricati.

Le fonti trattate sono tre, ognuna in una sotto-sezione dedicata:
- **4.1 INAIL** — infortuni sul lavoro
- **4.2 ISTAT ASIA** — imprese e addetti *(da sviluppare)*
- **4.3 INPS** — retribuzioni *(da sviluppare)*

### 4.1 INAIL — Infortuni sul lavoro

**Fonti di riferimento:**

*Dati infortuni (file da scaricare):*
- [Dati storici infortuni sul lavoro — pagina di download](https://dati.inail.it/portale/it/gli-open-data-inail/dati-storici/infortuni-sul-lavoro.html?page=1)
- [Calendario pubblicazioni](https://dati.inail.it/portale/it/gli-open-data-inail/calendario-pubblicazioni.html) — frequenza di aggiornamento (luglio / dicembre)
- Tracciato record: 25 campi per infortunio

*Tabelle di decodifica (tipologiche) — usate nella fase di pulizia (sez. 5):*
- [Elenco tipologiche INAIL](https://dati.inail.it/portale/it/dataset/infortuni-sul-lavoro/tipologiche.html)
  - Classificazione ATECO 2007 → decodifica settore
  - Province italiane (versione semestrale) → decodifica luogo → Regione/Provincia
  - Definizione amministrativa / Decisione istruttoria esito mortale → decodifica esito

#### 4.1.1 Valutazione della fonte

I dati INAIL sugli infortuni sono accessibili in due modi: tramite **API JSON** 
e tramite **file CSV (zip) scaricabili per regione e anno**. Prima di scegliere, 
entrambe le vie sono state testate empiricamente.

**API JSON — testata e scartata.**  
L'endpoint semestrale richiede i parametri `Regione`, `AnnoAccadimento` e 
`MeseAccadimento`. Testando in modo esaustivo tutte le combinazioni (anni 
2001–2025, tutti i mesi), è emerso che l'API:
- restituisce dati solo per gli ultimi mesi dell'anno (ottobre, novembre, 
  dicembre), non per l'anno intero;
- copre solo una finestra recente di circa 5 anni, non lo storico completo.

L'API fornisce quindi una porzione parziale dei dati e non è adatta a 
un'analisi storica. Si è scelto quindi di procedere con i file CSV (zip) 
scaricabili dal portale, la cui struttura interna viene analizzata nella 
sezione 4.1.2.

##### Evidenza: test di copertura dell'API

Il codice seguente interroga l'API per ogni combinazione anno/mese e registra 
quali mesi restituiscono dati. L'output conferma che l'API risponde solo per 
ottobre, novembre e dicembre, e solo per una finestra recente di anni — da cui 
la scelta di usare i file zip.

In [23]:
righe = []
for anno in range(2001, 2026):
    mesi_ok = []
    for mese in range(1, 13):
        url = f'https://dati.inail.it/api/OpenData/DatiConCadenzaSemestraleInfortuni?Regione=Lombardia&AnnoAccadimento={anno}&MeseAccadimento={mese}'
        esito = requests.get(url)
        if esito.status_code == 200:
            mesi_ok.append(mese)
    print(f'{anno}: mesi disponibili → {mesi_ok}')
    righe.append((anno, mesi_ok))

2001: mesi disponibili → []
2002: mesi disponibili → []
2003: mesi disponibili → []
2004: mesi disponibili → []
2005: mesi disponibili → []
2006: mesi disponibili → []
2007: mesi disponibili → []
2008: mesi disponibili → []
2009: mesi disponibili → []
2010: mesi disponibili → []
2011: mesi disponibili → []
2012: mesi disponibili → []
2013: mesi disponibili → []
2014: mesi disponibili → []
2015: mesi disponibili → []
2016: mesi disponibili → []
2017: mesi disponibili → []
2018: mesi disponibili → []
2019: mesi disponibili → []
2020: mesi disponibili → [10, 11, 12]
2021: mesi disponibili → [10, 11, 12]
2022: mesi disponibili → [10, 11, 12]
2023: mesi disponibili → [10, 11, 12]
2024: mesi disponibili → [10, 11, 12]
2025: mesi disponibili → []


L'API restituisce status 200 quando i dati sono presenti e 500 quando non lo sono. 
Nota: l'uso del 500 (ufficialmente "Internal Server Error") per segnalare l'assenza di dati non è conforme allo standard HTTP — ci si aspetterebbe un 404. Il comportamento è stato rilevato empiricamente testando la fonte.

In [43]:
# caso "dati non trovati" (mese vuoto)
r1 = requests.get('https://dati.inail.it/api/OpenData/DatiConCadenzaSemestraleInfortuni?Regione=Lombardia&AnnoAccadimento=2022&MeseAccadimento=1')
print(r1.status_code)

# caso "dati presenti" (mese pieno)
r2 = requests.get('https://dati.inail.it/api/OpenData/DatiConCadenzaSemestraleInfortuni?Regione=Lombardia&AnnoAccadimento=2022&MeseAccadimento=12')
print(r2.status_code)

500
200


#### 4.1.2 Analisi della struttura dei file CSV

Prima di scrivere lo script di download definitivo, si verifica empiricamente 
cosa contiene realmente un singolo file CSV scaricato dal portale: che periodo 
copre, se i dati al suo interno sono privi di duplicati, e come si relazionano 
tra loro le due versioni semestrali (01 e 02) dello stesso anno.

**creazione della def per scaricare e aprire i file**
questa definizione serve per standarizzare il download del file e verificare se è già stato scaricato, in tal caso lo apre soltanto

In [2]:
def scarico_apro_file(anno, semestre, regione):
    os.makedirs('dati_grezzi_inail', exist_ok=True)
    percorso_file = os.path.join('dati_grezzi_inail', f'{regione}_{semestre}_{anno}.zip')

    
    if os.path.exists(percorso_file):
        print(f'{percorso_file} esiste già, lo apro soltanto')
    else:
        url = f'https://dati.inail.it/opendata/downloads/dati_storici/{anno}/semestrale/{semestre}/daticoncadenzasemestraleinfortuni/zip/DatiConCadenzaSemestraleInfortuni{regione}_csv.zip'
        file = requests.get(url)
        print(file.status_code)
        with open(percorso_file, 'wb') as f:
            f.write(file.content)
    
    contenuto_zip = zipfile.ZipFile(percorso_file, 'r')
    df = pd.read_csv(contenuto_zip.open(f'DatiConCadenzaSemestraleInfortuni{regione}.csv'), sep=';', encoding='utf-8', keep_default_na=False)
    
    return df

**Download di un file campione** (Abruzzo, anno 2024, versione 02 — ottobre).  

In [3]:
df_abruzzo_2024_02 = scarico_apro_file(anno=2024, semestre='02', regione='Abruzzo')
df_abruzzo_2024_02

dati_grezzi_inail\Abruzzo_02_2024.zip esiste già, lo apro soltanto


,DataRilevazione,DataProtocollo,DataAccadimento,DataDefinizione,DataMorte,LuogoAccadimento,IdentificativoInfortunato,Genere,Eta,LuogoNascita,...,Indennizzo,DecisioneIstruttoriaEsitoMortale,GradoMenomazione,GiorniIndennizzati,IdentificativoDatoreLavoro,PosizioneAssicurativaTerritoriale,SettoreAttivitaEconomica,Gestione,GestioneTariffaria,GrandeGruppoTariffario
0,31/10/2024,10/10/2023,09/10/2023,18/11/2023,,69,33879477,F,25,ITAL,...,TE,ND,-1,33,-1,-1,ND,A,ND,ND
1,31/10/2024,04/05/2020,02/05/2020,16/05/2020,,69,9664912,F,55,ITAL,...,TE,ND,-1,5,4173909,13125854,Q 86,I,4,0
2,31/10/2024,15/02/2019,13/02/2019,15/02/2019,,66,10366804,F,36,ITAL,...,NE,ND,-1,0,8113548,10210134,Q 86,I,4,0
3,31/10/2024,16/01/2023,11/11/2022,17/01/2023,,69,15977682,F,48,ITAL,...,NE,ND,-1,0,4173909,4345837,Q 86,I,4,0
4,31/10/2024,12/02/2020,10/02/2020,21/02/2020,,69,481354,M,55,ITAL,...,TE,ND,-1,10,3636737,11420983,C 24,I,1,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63914,31/10/2024,07/02/2022,06/02/2022,07/02/2022,,69,33367428,F,29,Z129,...,NE,ND,-1,0,-1,-1,ND,I,ND,ND
63915,31/10/2024,24/05/2021,19/05/2021,03/08/2021,,68,33173040,M,39,ITAL,...,NE,ND,-1,0,3613161,264512,F 42,I,1,3
63916,31/10/2024,10/02/2022,09/02/2022,07/03/2022,,67,33372262,F,40,ITAL,...,NE,ND,-1,0,-1,-1,ND,S,ND,ND
63917,31/10/2024,26/01/2022,25/01/2022,03/03/2022,,67,33355374,F,46,ITAL,...,TE,ND,-1,8,3214593,4378170,Q 86,I,4,0


**Verifica: il file è privo di duplicati al suo interno?**  
`IdentificativoCaso` identifica il singolo evento di infortunio (a differenza 
di `IdentificativoInfortunato`, che identifica la persona e può ripetersi se 
la stessa persona ha avuto più infortuni in anni diversi). Se ogni 
`IdentificativoCaso` compare una sola volta, il file non contiene duplicati.

In [30]:
# info() mostra tipi di dato e conteggio valori non nulli per ogni colonna
df_abruzzo_2024_02.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63919 entries, 0 to 63918
Data columns (total 25 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   DataRilevazione                        63919 non-null  object
 1   DataProtocollo                         63919 non-null  object
 2   DataAccadimento                        63919 non-null  object
 3   DataDefinizione                        63919 non-null  object
 4   DataMorte                              63919 non-null  object
 5   LuogoAccadimento                       63919 non-null  int64 
 6   IdentificativoInfortunato              63919 non-null  int64 
 7   Genere                                 63919 non-null  object
 8   Eta                                    63919 non-null  int64 
 9   LuogoNascita                           63919 non-null  object
 10  ModalitaAccadimento                    63919 non-null  object
 11  ConSenzaMezzoTr

In [31]:
# conto quante volte compare ogni IdentificativoCaso
# se il valore massimo è 1, ogni caso è unico -> nessun duplicato nel file
verifica_casi_02 = df_abruzzo_2024_02.groupby(['IdentificativoCaso'])['IdentificativoCaso'].count().sort_values(ascending=False)
verifica_casi_02

IdentificativoCaso
25936482    1
22008423    1
22008538    1
22008611    1
22008698    1
           ..
22009271    1
22009255    1
22009208    1
22009162    1
22009092    1
Name: IdentificativoCaso, Length: 63919, dtype: int64

**Verifica: quale periodo di accadimento copre il file?**  
Si estrae l'anno dalla colonna `DataAccadimento` e se ne conta la distribuzione, 
per scoprire se il file "2024/02" contiene solo infortuni del 2024 oppure una 
finestra più ampia di anni. Il risultato verrà confrontato con lo stesso 
controllo fatto sul file `01` e su altri anni, per capire come si sovrappongono 
i file tra loro.

In [32]:
range_02_2024 = pd.to_datetime(df_abruzzo_2024_02['DataAccadimento'], format='%d/%m/%Y').dt.year.value_counts().sort_index()
range_02_2024

DataAccadimento
2019    13317
2020    11089
2021    11511
2022    15804
2023    12198
Name: count, dtype: int64

**Download del secondo file campione** (Abruzzo, anno 2024, versione 01 — aprile).  

In [33]:
df_abruzzo_2024_01 = scarico_apro_file(anno=2024, semestre='01', regione='Abruzzo')
df_abruzzo_2024_01

200


,DataRilevazione,DataProtocollo,DataAccadimento,DataDefinizione,DataMorte,LuogoAccadimento,IdentificativoInfortunato,Genere,Eta,LuogoNascita,...,Indennizzo,DecisioneIstruttoriaEsitoMortale,GradoMenomazione,GiorniIndennizzati,IdentificativoDatoreLavoro,PosizioneAssicurativaTerritoriale,SettoreAttivitaEconomica,Gestione,GestioneTariffaria,GrandeGruppoTariffario
0,30/04/2024,29/05/2023,22/05/2023,29/05/2023,,68,7634054,M,42,ITAL,...,NE,ND,-1,0,9442479,11722935,F 43,I,2,3
1,30/04/2024,30/03/2020,14/03/2020,09/05/2020,,68,16320938,F,62,ITAL,...,TE,ND,-1,42,3982905,4361490,Q 86,I,4,0
2,30/04/2024,03/02/2022,01/02/2022,21/04/2022,,66,33364283,F,44,ITAL,...,NE,ND,4,0,-1,-1,ND,S,ND,ND
3,30/04/2024,03/08/2023,31/07/2023,08/11/2023,,67,2045372,M,63,ITAL,...,NE,ND,-1,0,10005561,12517547,G 47,I,3,0
4,30/04/2024,14/11/2023,13/11/2023,18/01/2024,,69,6780487,M,66,ITAL,...,NE,ND,-1,0,-1,-1,ND,I,ND,ND
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63913,30/04/2024,06/10/2023,03/10/2023,01/01/1900,,67,33875843,M,58,ITAL,...,NE,ND,-1,0,10756351,13577992,F 41,I,2,3
63914,30/04/2024,09/04/2021,08/04/2021,13/07/2021,,69,33141517,M,21,ITAL,...,NE,ND,-1,0,9813372,12251864,N 82,I,3,0
63915,30/04/2024,21/04/2021,20/04/2021,11/05/2021,,67,33148845,F,24,ITAL,...,NE,ND,-1,0,3819958,12893340,Q 86,I,3,0
63916,30/04/2024,28/09/2021,24/09/2021,02/11/2021,,67,33253997,M,13,ITAL,...,NE,ND,-1,0,-1,-1,ND,S,ND,ND


**Verifica: il file è privo di duplicati al suo interno?**  
`IdentificativoCaso` identifica il singolo evento di infortunio (a differenza 
di `IdentificativoInfortunato`, che identifica la persona e può ripetersi se 
la stessa persona ha avuto più infortuni in anni diversi). Se ogni 
`IdentificativoCaso` compare una sola volta, il file non contiene duplicati.

In [34]:
df_abruzzo_2024_01.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63918 entries, 0 to 63917
Data columns (total 25 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   DataRilevazione                        63918 non-null  object
 1   DataProtocollo                         63918 non-null  object
 2   DataAccadimento                        63918 non-null  object
 3   DataDefinizione                        63918 non-null  object
 4   DataMorte                              63918 non-null  object
 5   LuogoAccadimento                       63918 non-null  int64 
 6   IdentificativoInfortunato              63918 non-null  int64 
 7   Genere                                 63918 non-null  object
 8   Eta                                    63918 non-null  int64 
 9   LuogoNascita                           63918 non-null  object
 10  ModalitaAccadimento                    63918 non-null  object
 11  ConSenzaMezzoTr

In [35]:
# conto quante volte compare ogni IdentificativoCaso
# se il valore massimo è 1, ogni caso è unico -> nessun duplicato nel file
verifica_casi_01 = df_abruzzo_2024_01.groupby('IdentificativoCaso')['IdentificativoCaso'].count().sort_values(ascending=False)
verifica_casi_01

IdentificativoCaso
25668869    1
22008423    1
22008538    1
22008611    1
22008698    1
           ..
22009255    1
22009208    1
22009162    1
22009092    1
22009066    1
Name: IdentificativoCaso, Length: 63918, dtype: int64

**Verifica: quale periodo di accadimento copre il file?**  
Si estrae l'anno dalla colonna `DataAccadimento` e se ne conta la distribuzione, 
per scoprire se il file "2024/02" contiene solo infortuni del 2024 oppure una 
finestra più ampia di anni. Il risultato verrà confrontato con lo stesso 
controllo fatto sul file `01` e su altri anni, per capire come si sovrappongono 
i file tra loro.

In [36]:
range_01_2024 = pd.to_datetime(df_abruzzo_2024_01['DataAccadimento'],format='%d/%m/%Y').dt.year.value_counts().sort_index()
range_01_2024

DataAccadimento
2019    13320
2020    11093
2021    11512
2022    15801
2023    12192
Name: count, dtype: int64

**Conclusione: confronto tra le versioni 01 e 02 dello stesso anno (2024)**

I due file sono stati confrontati su tre aspetti:
- **Duplicati interni**: nessuno dei due file contiene infortuni duplicati — 
  in entrambi, ogni `IdentificativoCaso` compare esattamente una volta.
- **Range di accadimento coperto**: entrambi i file coprono la stessa finestra 
  di 5 anni (2019-2023), confermando che `01` e `02` fotografano lo stesso 
  periodo storico, a sei mesi di distanza l'uno dall'altro.
- **Sovrapposizione dei casi**: il 99,98% degli `IdentificativoCaso` è presente 
  in entrambe le versioni (63.906 su 63.918); 12 casi risultano solo nel `01` 
  e 13 solo nel `02` — una differenza minima, compatibile con normali 
  aggiornamenti amministrativi tra le due rilevazioni.

**Implicazione:** le due versioni sono, ai fini di questa analisi, praticamente 
equivalenti. La scelta di usare sempre la versione `02` (ottobre, più 
consolidata) resta valida e viene mantenuta come criterio per garantire 
coerenza tra tutti gli anni della serie storica.

**Eccezione per il 2025.** La versione `02` (ottobre) del 2025 non è ancora 
pubblicata è invece disponibile la versione `01` (aprile), con dati reali (62.415 righe, accadimento 2020-2024).

Data la differenza trascurabile misurata tra `01` e `02` per lo stesso anno 
(sotto lo 0,3% sui conteggi, 99,98% di sovrapposizione sugli `IdentificativoCaso`), 
si sceglie di fare eccezione per il solo 2025 e usare la versione `01` al posto 
della `02`, non ancora disponibile. Per tutti gli altri anni della serie 
resta valida la versione `02`.

**Download del terzo file campione** (Abruzzo, anno 2014, versione 02 — ottobre).  

In [6]:
df_abruzzo_2014_02 = scarico_apro_file(anno=2014, semestre='02', regione='Abruzzo')
df_abruzzo_2014_02

il file Abruzzo_02_2014.zip esiste già, lo apro soltanto


,DataRilevazione,DataProtocollo,DataAccadimento,DataDefinizione,DataMorte,LuogoAccadimento,IdentificativoInfortunato,Genere,Eta,LuogoNascita,...,Indennizzo,DecisioneIstruttoriaEsitoMortale,GradoMenomazione,GiorniIndennizzati,IdentificativoDatoreLavoro,PosizioneAssicurativaTerritoriale,SettoreAttivitaEconomica,Gestione,GestioneTariffaria,GrandeGruppoTariffario
0,31/10/2014,27/10/2010,25/10/2010,26/01/2011,,67,9406744,M,46,ITAL,...,NE,ND,-1,0,-1,-1,ND,I,ND,ND
1,31/10/2014,27/04/2012,26/04/2012,22/05/2012,,67,10213856,M,22,Z129,...,TE,ND,-1,15,-1,-1,ND,A,ND,ND
2,31/10/2014,30/10/2012,27/10/2012,03/12/2012,,69,2945026,M,70,ITAL,...,TE,ND,3,35,-1,-1,ND,A,ND,ND
3,31/10/2014,27/05/2010,14/05/2010,07/07/2010,,68,9244516,M,7,ITAL,...,NE,ND,-1,0,-1,-1,ND,S,ND,ND
4,31/10/2014,06/03/2012,05/03/2012,22/03/2013,,66,4973024,F,45,ITAL,...,NE,ND,-1,0,-1,-1,ND,I,ND,ND
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98696,31/10/2014,07/08/2012,31/07/2012,07/09/2012,,66,10336821,M,50,ITAL,...,TE,ND,-1,1,3558897,5068136,C 21,I,1,0
98697,31/10/2014,07/07/2009,30/06/2009,08/04/2011,,66,8770760,M,60,ITAL,...,TE,ND,-1,12,8134770,10250065,ND,I,ND,ND
98698,31/10/2014,14/11/2013,13/11/2013,15/09/2014,,66,11144581,F,40,ITAL,...,NE,ND,-1,0,6155945,10818191,G 47,I,3,0
98699,31/10/2014,15/05/2012,09/05/2012,12/06/2012,,69,9541695,M,55,ITAL,...,NE,ND,-1,0,7634466,9551049,N 81,I,3,0


**Verifica: il file è privo di duplicati al suo interno?**  
`IdentificativoCaso` identifica il singolo evento di infortunio (a differenza 
di `IdentificativoInfortunato`, che identifica la persona e può ripetersi se 
la stessa persona ha avuto più infortuni in anni diversi). Se ogni 
`IdentificativoCaso` compare una sola volta, il file non contiene duplicati.

In [40]:
df_abruzzo_2014_02.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98701 entries, 0 to 98700
Data columns (total 25 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   DataRilevazione                        98701 non-null  object
 1   DataProtocollo                         98701 non-null  object
 2   DataAccadimento                        98701 non-null  object
 3   DataDefinizione                        98701 non-null  object
 4   DataMorte                              98701 non-null  object
 5   LuogoAccadimento                       98701 non-null  int64 
 6   IdentificativoInfortunato              98701 non-null  int64 
 7   Genere                                 98701 non-null  object
 8   Eta                                    98701 non-null  int64 
 9   LuogoNascita                           98701 non-null  object
 10  ModalitaAccadimento                    98701 non-null  object
 11  ConSenzaMezzoTr

In [45]:
# conto quante volte compare ogni IdentificativoCaso
# se il valore massimo è 1, ogni caso è unico -> nessun duplicato nel file
verifica_casi_02_2014 = df_abruzzo_2014_02.groupby('IdentificativoCaso')['IdentificativoCaso'].count().sort_values(ascending=False)
verifica_casi_02_2014

IdentificativoCaso
17338666    1
11711844    1
11711876    1
11711878    1
11711879    1
           ..
11711925    1
11711924    1
11711923    1
11711922    1
11711921    1
Name: IdentificativoCaso, Length: 98701, dtype: int64

**Verifica: quale periodo di accadimento copre il file?**  
Si estrae l'anno dalla colonna `DataAccadimento` e se ne conta la distribuzione, 
per scoprire se il file "2014/02" contiene solo infortuni del 2014 oppure una 
finestra più ampia di anni. Il risultato verrà confrontato su altri anni, per capire come si sovrappongono 
i file tra loro.

In [7]:
range_02_2014 = pd.to_datetime(df_abruzzo_2014_02['DataAccadimento'],format='%d/%m/%Y').dt.year.value_counts().sort_index()
range_02_2014

DataAccadimento
2009    21548
2010    21708
2011    20455
2012    18320
2013    16670
Name: count, dtype: int64

#### 4.1.3 Decisione e perimetro

**La regola verificata.** Ogni file scaricabile copre una finestra mobile di 
5 anni di accadimento: il file `{anno}` contiene gli infortuni accaduti tra 
`{anno-5}` e `{anno-1}`. La regola è stata confermata empiricamente su due 
fronti:
- confrontando le due versioni dello stesso anno (2024/01 vs 2024/02): 
  stessa finestra (2019-2023), stessa struttura, 99,98% degli 
  `IdentificativoCaso` in comune — le due versioni sono equivalenti, e per 
  coerenza si userà sempre la versione `02` (ottobre), quando disponibile;
- verificando il file più vecchio disponibile in CSV (2014/02): stessa 
  struttura tabellare, finestra 2009-2013, esattamente quella prevista dalla 
  regola.

Essendo confermata sia su un file recente sia sul più vecchio disponibile, la 
regola viene applicata con sicurezza per scegliere i file necessari a coprire 
l'intero perimetro, senza dover scaricare e verificare ogni singolo anno di 
release uno per uno.

**I file scelti.** Applicando la regola, tre file bastano a coprire l'intero 
storico disponibile senza sovrapposizioni residue, tenendo per ogni anno solo 
la fetta proveniente dal file più recente disponibile:

| File | Finestra coperta | Anni effettivamente tenuti |
|------|------------------|------------------------------|
| `2019/02` | 2014-2018 | 2014, 2015, 2016, 2017, 2018 |
| `2024/02` | 2019-2023 | solo 2019 (2020-2023 sovrascritti da `2025/01`, più recente) |
| `2025/01` | 2020-2024 | 2020, 2021, 2022, 2023, 2024 |

**Eccezione sul 2025/01.** Per l'anno di release 2025 si usa la versione `01` 
(aprile) al posto della `02` (ottobre), perché quest'ultima non è ancora 
pubblicata (uscirà a dicembre 2026, secondo il calendario INAIL). La 
differenza misurata tra le due versioni per lo stesso anno è trascurabile 
(sotto lo 0,3% sui conteggi), quindi l'eccezione è accettabile e non 
compromette la coerenza della serie.

**Perimetro finale:** accadimento **2014-2024**, 11 anni, nessuna 
sovrapposizione, nessun buco.

> **Nota — dataset a cadenza mensile.** INAIL pubblica anche una versione a 
> cadenza mensile degli stessi dati (`DatiConCadenzaMensileInfortuni{Regione}`), 
> più aggiornata di quella semestrale (dati fino a maggio 2026 al momento della 
> scrittura). Non viene usata in questo progetto per due motivi: 
> - (1) copre solo l'anno corrente e lo stesso periodo dell'anno precedente per confronto YoY (es. gennaio-maggio 2025 vs gennaio-maggio 2026), non uno storico continuo; 
> - (2) mancano 7 campi presenti nella versione semestrale, tra cui `GiorniIndennizzati`, necessario per la stima dei costi (una delle domande di analisi del progetto). Potrebbe tornare utile in futuro per un modulo "ultimi mesi" nella dashboard, separato dall'analisi storica.

#### 4.1.4 Download del dataset completo

Applicando la regola e la scelta dei file stabilite in 4.1.3, si scaricano ora 
i tre file (`2019/02`, `2024/02`, `2025/01`) per **tutte le regioni** 
disponibili, non solo per Abruzzo.

**Nota sui nomi delle regioni.** L'elenco delle 20 regioni è quello del menu 
a tendina del sito. Per le regioni con nome composto (spazi, trattini o 
apostrofi — es. `Valle d'Aosta`, `Emilia-Romagna`, `Friuli-Venezia Giulia`), 
il formato richiesto dall'URL è stato verificato empiricamente su tre casi 
(`ValledAosta`, `EmiliaRomagna`, `FriuliVeneziaGiulia`, tutti risposti con 
`200` e file reale): la regola è togliere spazi, trattini e apostrofi, 
lasciando il resto invariato.

I dataframe risultanti (20 regioni × 3 combinazioni = 60 in totale) vengono 
salvati in un dizionario, indicizzato per `(regione, anno, semestre)`, così 
da poterli recuperare singolarmente nei passaggi successivi (filtro per anno 
e concatenazione).

**Verifica dei nomi composti**

In [4]:
anno = 2024
semestre = '02'
regione = 'FriuliVeneziaGiulia'
url = f'https://dati.inail.it/opendata/downloads/dati_storici/{anno}/semestrale/{semestre}/daticoncadenzasemestraleinfortuni/zip/DatiConCadenzaSemestraleInfortuni{regione}_csv.zip'
r = requests.get(url)
print(r.status_code, r.headers.get('Content-Disposition'))


200 attachment; filename="DatiConCadenzaSemestraleInfortuniFriuliVeneziaGiulia_csv.zip"


**Download di tutti i file necessari**

In [4]:
combinazioni = [(2019, '02'), (2024, '02'), (2025, '01')]
regioni = [
    'Abruzzo', 'Basilicata', 'Calabria', 'Campania', 'EmiliaRomagna',
    'FriuliVeneziaGiulia', 'Lazio', 'Liguria', 'Lombardia', 'Marche',
    'Molise', 'Piemonte', 'Puglia', 'Sardegna', 'Sicilia', 'Toscana',
    'TrentinoAltoAdige', 'Umbria', 'ValledAosta', 'Veneto'
]

risultati = {}   # qui salviamo ogni dataframe, con una chiave che identifica regione/anno/semestre

for reg in regioni:
    for anno, semestre in combinazioni:   # spacchetto la tupla direttamente nel for
        df = scarico_apro_file(anno=anno, semestre=semestre, regione=reg)
        risultati[(reg, anno, semestre)] = df

dati_grezzi_inail\Abruzzo_02_2019.zip esiste già, lo apro soltanto
dati_grezzi_inail\Abruzzo_02_2024.zip esiste già, lo apro soltanto
dati_grezzi_inail\Abruzzo_01_2025.zip esiste già, lo apro soltanto
dati_grezzi_inail\Basilicata_02_2019.zip esiste già, lo apro soltanto
dati_grezzi_inail\Basilicata_02_2024.zip esiste già, lo apro soltanto
dati_grezzi_inail\Basilicata_01_2025.zip esiste già, lo apro soltanto
dati_grezzi_inail\Calabria_02_2019.zip esiste già, lo apro soltanto
dati_grezzi_inail\Calabria_02_2024.zip esiste già, lo apro soltanto
dati_grezzi_inail\Calabria_01_2025.zip esiste già, lo apro soltanto
dati_grezzi_inail\Campania_02_2019.zip esiste già, lo apro soltanto
dati_grezzi_inail\Campania_02_2024.zip esiste già, lo apro soltanto
dati_grezzi_inail\Campania_01_2025.zip esiste già, lo apro soltanto
dati_grezzi_inail\EmiliaRomagna_02_2019.zip esiste già, lo apro soltanto
dati_grezzi_inail\EmiliaRomagna_02_2024.zip esiste già, lo apro soltanto
dati_grezzi_inail\EmiliaRomagna_01_

**verifica su quanti file/database ho scaricato**

In [5]:
len(risultati)

60

In [9]:
risultati[('Veneto', 2024, '02')].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 371695 entries, 0 to 371694
Data columns (total 25 columns):
 #   Column                                 Non-Null Count   Dtype 
---  ------                                 --------------   ----- 
 0   DataRilevazione                        371695 non-null  object
 1   DataProtocollo                         371695 non-null  object
 2   DataAccadimento                        371695 non-null  object
 3   DataDefinizione                        371695 non-null  object
 4   DataMorte                              371695 non-null  object
 5   LuogoAccadimento                       371695 non-null  int64 
 6   IdentificativoInfortunato              371695 non-null  int64 
 7   Genere                                 371695 non-null  object
 8   Eta                                    371695 non-null  int64 
 9   LuogoNascita                           371695 non-null  object
 10  ModalitaAccadimento                    371695 non-null  object
 11  

#### 4.1.5 Costruzione del dataset INAIL finale

Applicando il perimetro deciso in 4.1.3, ogni dataframe scaricato in 4.1.4 
viene filtrato per tenere solo la fetta di anni di propria competenza (evitando 
sovrapposizioni tra file), poi tutti i pezzi vengono concatenati in un unico 
dataframe.

**Risultato:** `df_inail`, **6.901.352 righe**, 25 colonne, storico di 
accadimento **2014-2024** completo — undici anni consecutivi, nessun buco, 
nessuna sovrapposizione residua. Il dataset viene salvato su disco per non 
dover ripetere il processo di filtro e concatenazione ad ogni riapertura del 
notebook.

In [34]:
# mappa combinazione (anno, semestre) -> range di anni di accadimento da tenere,
# secondo la decisione presa in 4.1.3 (una fetta diversa per ogni file, senza sovrapposizioni)
anni_da_tenere = {
    (2019, '02'): range(2014, 2019),   # tengo tutto: 2014, 2015, 2016, 2017, 2018
    (2024, '02'): range(2019, 2020),   # tengo solo il 2019 (2020-2023 coperti meglio dal 2025/01)
    (2025, '01'): range(2020, 2025),   # tengo tutto: 2020, 2021, 2022, 2023, 2024
}

# lista vuota dove accumulo i 60 dataframe già filtrati, pronti per la concatenazione finale
df_filtrati = []

# scorro tutte le 60 combinazioni salvate in risultati (20 regioni x 3 file)
# .items() mi dà, per ogni giro, sia la chiave (reg, anno, semestre) sia il dataframe corrispondente
for (reg, anno, semestre), df in risultati.items():

    # converto la colonna DataAccadimento (testo) in vere date, poi estraggo solo l'anno
    anno_accadimento = pd.to_datetime(df['DataAccadimento'], format='%d/%m/%Y').dt.year

    # recupero dal dizionario il range di anni da tenere per QUESTA combinazione (anno, semestre)
    # non uso reg qui, perché il range non dipende dalla regione: è lo stesso per tutte e 20
    range_da_tenere = anni_da_tenere[(anno, semestre)]

    # filtro il dataframe: tengo solo le righe il cui anno di accadimento rientra nel range scelto
    df_filtrato = df[anno_accadimento.isin(range_da_tenere)]

    # aggiungo il dataframe filtrato in fondo alla lista (uno per ogni giro del ciclo)
    df_filtrati.append(df_filtrato)

# concateno tutti i 60 pezzi filtrati in un unico dataframe, impilandoli uno sopra l'altro
# ignore_index=True evita indici duplicati (altrimenti ogni pezzo riparte da 0)
df_inail = pd.concat(df_filtrati, ignore_index=True)

# salvo il dataset finale su disco (separatore ';', niente colonna indice extra)
# così non serve ripetere il filtro + concatenazione ad ogni riapertura del notebook
df_inail.to_csv('dati_grezzi_inail/df_inail_finale.csv', index=False, sep=';')

# controllo rapido: righe totali e numero di colonne del dataset finale
df_inail.shape

(6901352, 25)